In [4]:
%use kandy

In [8]:
val Tmax =  125
val Tamb = 40

val Rth_jc_igbt     = 0.88
val Rth_jc_diodo    = 1.78
val Rth_cd          = 0.2
val Rth_da          = 0.9
val Rth_ca          = Rth_cd + Rth_da

val Pmax_igbt = (Tmax - Tamb) / (Rth_jc_igbt + Rth_ca)
val Pmax_diodo = (Tmax - Tamb) / (Rth_jc_diodo + Rth_ca)

val Req  = 1 / (6 / Rth_jc_igbt + 6 / Rth_jc_diodo)
val Ptot = (Tmax - Tamb) / (Req + Rth_ca)
val Pmax_igbt1 = Req / Rth_jc_igbt * Ptot
val Pmax_diodo1 = Req / Rth_jc_diodo * Ptot

println("Potencia Maxima IGBT: ${String.format("%.2f", Pmax_igbt)} W")
println("Potencia Maxima IGBT1: ${String.format("%.2f", Pmax_igbt1)} W")
println("Potencia Maxima Diodo: ${String.format("%.2f", Pmax_diodo)} W")
println("Potencia Maxima Diodo1: ${String.format("%.2f", Pmax_diodo1)} W")
println("Potencia Maxima Total: ${String.format("%.2f", Ptot)} W")

Potencia Maxima IGBT: 42,93 W
Potencia Maxima IGBT1: 7,91 W
Potencia Maxima Diodo: 29,51 W
Potencia Maxima Diodo1: 3,91 W
Potencia Maxima Total: 70,94 W


## Parametros del Puente

In [9]:
val Vce_sat     = 1.55f //V
val Vce_bloq    = 300   //V
val Vce0        = 0.85f    //V
val rce_igbt    = (Vce_sat - Vce0) / 50//Ohm
val Vf0         = 0.4428 //V
val rce_diodo   = (1.7 - Vf0) / 50

val Vcc = 300
val Vac_rms = 28
val Vac_max_rms = 0.577 * Vcc / sqrt(2f)
val m = Vac_rms / Vac_max_rms
val f_sw = 10000
val f_red = 50

println("Vce0\t\t= ${String.format("%.2f",Vce0)} V ")
println("rce_igbt\t= ${String.format("%.2f",rce_igbt)} Ohm ")
println("Vf0\t\t\t= ${String.format("%.2f",Vf0)} V ")
println("rce_diodo\t= ${String.format("%.3f",rce_diodo)} Ohm ")
println("m\t\t\t= ${String.format("%.3f", m)}")

Vce0		= 0,85 V 
rce_igbt	= 0,01 Ohm 
Vf0			= 0,44 V 
rce_diodo	= 0,025 Ohm 
m			= 0,229


## Formulas para calculo de perdidas en conmutacion

In [12]:
fun calcular_energia_funciones_lineales(Vi: Float, Vf: Float, Ii: Float, If: Float, deltaT: Double): Double {
    val deltaV = Vf - Vi
    val deltaI = If - Ii
    return ( deltaV * deltaI / 3 + (Vi*deltaI + Ii*deltaV) / 2 + Vi * Ii ) * deltaT
}

fun calcular_potencia_encendido(il_peak: Float): Double {
    var E_ciclo = 0.0
    val tr = 0.6 * 10f.pow(-6)
    val cant_puntos = (0.5 * f_sw / f_red).roundToInt()
    for(i in (0..cant_puntos)) {
        val theta = PI / cant_puntos * i
        val Ic = il_peak * sin(theta)
        E_ciclo += calcular_energia_funciones_lineales(
            Vi = 0.9f * Vce_bloq,
            Vf = 0.1f * Vce_bloq,
            Ii = 0.1f * Ic.toFloat(),
            If = Ic.toFloat(),
            deltaT = tr
        )
    }
    return f_red * E_ciclo
}

fun calcular_potencia_apagado(il_peak: Float): Double {
    var E_ciclo = 0.0
    val tr = 1.2 * 10f.pow(-6)
    val cant_puntos = (0.5 * f_sw / f_red).roundToInt()
    for(i in (0..cant_puntos)) {
        val theta = PI / cant_puntos * i
        val Ic = il_peak * sin(theta)
        E_ciclo += calcular_energia_funciones_lineales(
            Vi = 0.1f * Vce_bloq, //TODO deberia ser Vce_sat
            Vf = 0.1f * Vce_bloq,
            Ii = 0.9f * Ic.toFloat(),
            If = 0.2f * Ic.toFloat(),
            deltaT = tr
        )
    }
    return f_red * E_ciclo
}

fun calcular_potencia_encendido_alt(il_peak: Float): Double {
    var E_ciclo = 0.0
    val cant_puntos = (0.5 * f_sw / f_red).roundToInt()
    for(i in (0..cant_puntos)) {
        val theta = PI / cant_puntos * i
        val Ic = il_peak * sin(theta)
        E_ciclo += 0.0000001448706351 * Vce_bloq * Ic
    }
    return f_red * E_ciclo
}

fun calcular_potencia_apagado_alt(il_peak: Float): Double {
    var E_ciclo = 0.0
    val cant_puntos = (0.5 * f_sw / f_red).roundToInt()
    for(i in (0..cant_puntos)) {
        val theta = PI / cant_puntos * i
        val Ic = il_peak * sin(theta)
        E_ciclo += 0.0000002921576001 * Vce_bloq * Ic
    }
    return f_red * E_ciclo
}


## Formulas para calculo de perdidas en conduccion
Se utilizan las formulas propuestas en el trabajo "Semiconductor Losses in Voltage Source and Current Source IGBT Converters Based on Analytical Derivation". Si bien trabajamos con modulacion SVPWM, las formulas en cuestion corresponden a modulacion senoidal PWM, que, segun indica el trabajo, sirven como una buena aproximacion.
### Potencia de conduccion disipada por el IGBT
Si se considera $\cos(\phi) \approx 1$ resulta:
$$ P_{C,IGBT}=\frac{(\frac{\pi}{4} + \frac{2}{3} m) rce}{2\pi} il_{peak}^2   +   \frac{(1 + \frac{\pi}{4} * m) * Vce0}{2\pi} il_{peak} $$
### Potencia de conduccion disipada por el diodo parásito
Si se considera $\cos(\phi) \approx 1$ resulta:
$$ P_{C,Diodo}=\frac{(\frac{\pi}{4} - \frac{2}{3} m) rce}{2\pi} il_{peak}^2   +   \frac{(1 - \frac{\pi}{4} * m) * Vf0}{2\pi} il_{peak} $$

In [11]:
fun calcular_potencia_conduccion_igbt(il_peak: Float): Double {
    return (PI/4 + 2/3 * m) * rce_igbt * il_peak.pow(2) / (2*PI) + (1 + PI/4 * m) * Vce0 * il_peak / (2*PI)
}

fun calcular_potencia_conduccion_diodo(il_peak: Float): Double {
    return (PI/4 - 2/3 * m) * rce_diodo * il_peak.pow(2) / (2*PI) + (1 - PI/4 * m) * Vf0 * il_peak / (2*PI)
}

# Plots

In [9]:
val corriente = List(101) { i -> i}
val p_cond_igbt     = corriente.map { i -> calcular_potencia_conduccion_igbt(i.toFloat())}
val p_cond_diodo    = corriente.map { i -> calcular_potencia_conduccion_diodo(i.toFloat())}

val p_on            = corriente.map { i -> calcular_potencia_encendido(i.toFloat())}
val p_on_alt           = corriente.map { i -> calcular_potencia_encendido_alt(i.toFloat())}

val p_off           = corriente.map { i -> calcular_potencia_apagado(i.toFloat())}
val p_off_alt           = corriente.map { i -> calcular_potencia_apagado_alt(i.toFloat())}

run {
    val data = mapOf(
        "corriente" to corriente + corriente + corriente + corriente + corriente + corriente,
        "potencia" to p_cond_igbt + p_cond_diodo + p_on + p_on_alt + p_off + p_off_alt,
        "legends" to List(corriente.size) { "Conduccion IGBT" } + List(corriente.size) { "Conduccion Diodo" } + List(corriente.size) { "Encendido" } + List(corriente.size) { "Encendido Alternativo" } + List(corriente.size) { "Apagado" } + List(corriente.size) { "Apagado Alternativo" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("corriente") { axis.name = "Corriente de Pico [A]"}
                y("potencia") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Cu6HsP"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"corriente",
"y":"potencia",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"potencia":[0.0,0.1705996992508716,0.34444939833596777,0.5215490972552885,0.7018987960088339,0.8854984945966038,1.0723481930185983,1.2624478912748174,1.455797589365261,1.6523972872899295,1.8522469850488221,2.0553466826419395,2.2616963800692815,2.471296077330848,2.6841457744266393,2.900245471356655,3.1195951681208953,3.3421948647193607,3.56804456115205,3.7971442574189638,4.029493953520102,4.265093649455466,4.5039433452250535,4.746043040828866,4.991392736266903,5.239992431539164,5.4918421266456505,5.746941821586361,6.0052915163612965,6.266891210970456,6.531740905413841,6.79984059969145,7.071190293803284,7.345789987749342,7.6236396815296255,7.904739375144133,8.189089068592864,8.47668876187582,8.767538454993002,9.061638147944407,9.358987840730038,9.659587533349892,9.963437225803972,10.270536918092276,10.580886610214804,10.894486302171558,11.211335993962535,11.531435685587738,11.854785377047165,12.181385068340816,12.511234759468692,12.844334450430793,13.180684141227118,13.520283831857668,13.863133522322443,14.209233212621442,14.558582902754665,14.911182592722113,15.267032282523786,15.626131972159683,15.988481661629805,16.354081350934152,16.722931040072723,17.09503072904552,17.47038041785254,17.848980106493784,18.230829794969253,18.615929483278947,19.004279171422866,19.39587885940101,19.790728547213376,20.18882823485997,20.590177922340786,20.994777609655827,21.402627296805093,21.813726983788584,22.2280766706063,22.64567635725824,23.066526043744403,23.490625730064792,23.917975416219406,24.348575102208244,24.782424788031307,25.219524473688594,25.659874159180106,26.103473844505842,26.550323529665803,27.00042321465999,27.4537728994884,27.910372584151034,28.370222268647893,28.833321952978977,29.299671637144286,29.76927132114382,30.242121004977577,30.71822068864556,31.197570372147766,31.680170055484197,32.16601973865485,32.655119421659734,33.14746910449884,0.0,0.060955063327632855,0.1281961266552657,0.20172318998289857,0.2815362533105314,0.3676353166381643,0.4600203799657971,0.55869144329343,0.6636485066210628,0.7748915699486956,0.8924206332763285,1.0162356966039614,1.1463367599315941,1.2827238232592268,1.42539688658686,1.5743559499144926,1.7296010132421256,1.8911320765697583,2.058949139897391,2.233052203225024,2.413441266552657,2.6001163298802896,2.7930773932079225,2.992324456535555,3.197857519863188,3.409676583190821,3.6277816465184536,3.8521727098460863,4.08284977317372,4.319812836501352,4.563061899828985,4.812596963156618,5.068418026484251,5.330525089811884,5.5989181531395165,5.873597216467148,6.154562279794781,6.441813343122415,6.735350406450047,7.03517346977768,7.341282533105312,7.653677596432947,7.9723586597605784,8.297325723088212,8.628578786415845,8.966117849743476,9.30994291307111,9.660053976398743,10.016451039726375,10.379134103054009,10.748103166381641,11.123358229709275,11.504899293036905,11.89272635636454,12.286839419692171,12.687238483019806,13.093923546347439,13.506894609675072,13.926151673002703,14.351694736330337,14.783523799657969,15.221638862985603,15.666039926313234,16.116726989640867,16.5737000529685,17.03695911629613,17.506504179623764,17.9823352429514,18.464

In [10]:
val p_cond_diodo    = corriente.map { i -> calcular_potencia_conduccion_diodo(i.toFloat())}
val p_total_diodo = p_cond_diodo

run {
    val data = mapOf(
        "corriente" to corriente,
        "potencia" to p_total_diodo,
    )  // Combine data into a map

    plot(data) { // Begin plotting
        hLine {
            yIntercept.constant(Pmax_diodo)
            color = Color.RED
            type = LineType.DASHED
        }
        line {
            x("corriente") { axis.name = "Corriente de Pico [A]"}
            y("potencia") { axis.name = "Potencia [W]" }
            color = Color.YELLOW
        }
        layout { // Set plot layout
            title = "Disipacion de potencia - DIODO" // Add title
            size = 1300 to 500 // Plot dimension settings
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="aiuJpa"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia - DIODO"
},
"mapping":{
},
"data":{
"potencia":[0.0,0.060955063327632855,0.1281961266552657,0.20172318998289857,0.2815362533105314,0.3676353166381643,0.4600203799657971,0.55869144329343,0.6636485066210628,0.7748915699486956,0.8924206332763285,1.0162356966039614,1.1463367599315941,1.2827238232592268,1.42539688658686,1.5743559499144926,1.7296010132421256,1.8911320765697583,2.058949139897391,2.233052203225024,2.413441266552657,2.6001163298802896,2.7930773932079225,2.992324456535555,3.197857519863188,3.409676583190821,3.6277816465184536,3.8521727098460863,4.08284977317372,4.319812836501352,4.563061899828985,4.812596963156618,5.068418026484251,5.330525089811884,5.5989181531395165,5.873597216467148,6.154562279794781,6.441813343122415,6.735350406450047,7.03517346977768,7.341282533105312,7.653677596432947,7.9723586597605784,8.297325723088212,8.628578786415845,8.966117849743476,9.30994291307111,9.660053976398743,10.016451039726375,10.379134103054009,10.748103166381641,11.123358229709275,11.504899293036905,11.89272635636454,12.286839419692171,12.687238483019806,13.093923546347439,13.506894609675072,13.926151673002703,14.351694736330337,14.783523799657969,15.221638862985603,15.666039926313234,16.116726989640867,16.5737000529685,17.03695911629613,17.506504179623764,17.9823352429514,18.464452306279032,18.952855369606663,19.447544432934293,19.94851949626193,20.455780559589563,20.969327622917195,21.48916068624483,22.01527974957246,22.547684812900094,23.086375876227727,23.63135293955536,24.18261600288299,24.740165066210622,25.30400012953826,25.874121192865893,26.45052825619352,27.033221319521157,27.622200382848785,28.217465446176423,28.819016509504053,29.42685357283169,30.040976636159318,30.66138569948695,31.288080762814587,31.921061826142218,32.56032888946985,33.20588195279748,33.85772101612512,34.51584607945274,35.180257142780384,35.850954206108014,36.52793726943565,37.211206332763275],
"corriente":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0]
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"yintercept":29.51388888888889,
"color":"#ee6666",
"linetype":"dashed",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"hline",
"data":{
}
},{
"mapping":{
"x":"corriente",
"y":"potencia"
},
"stat":"identity",
"color":"#fac858",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"corriente"
},{
"type":"float",
"column":"potencia"
}]
},
"spec_id":"5"
};
 var containerDiv = document.getElementById("aiuJpa");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1300.0,
 height: 500.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 <

In [11]:
val p_total         = p_cond_igbt
    .zip(p_on_alt) { a,b -> a+b }
    .zip(p_off_alt){ a,b -> a+b }
run {
    val data = mapOf(
        "corriente" to corriente,
        "potencia" to p_total,
    )  // Combine data into a map

    plot(data) { // Begin plotting
        hLine {
            yIntercept.constant(Pmax_igbt)
            color = Color.RED
            type = LineType.DASHED
        }
        line {
            x("corriente") { axis.name = "Corriente de Pico [A]"}
            y("potencia") { axis.name = "Potencia [W]" }
            color = Color.YELLOW
        }
        layout { // Set plot layout
            title = "Disipacion de potencia" // Add title
            size = 1300 to 500 // Plot dimension settings
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="VQJmOP"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
"potencia":[0.0,0.587896597985766,1.1790431958057566,1.7734397934599717,2.3710863909484114,2.9719829882710758,3.5761295854279647,4.1835261824190795,4.794172779244416,5.40806937590398,6.025215972397766,6.645612568725779,7.269259164888014,7.896155760884476,8.526302356715163,9.15969895238007,9.796345547879206,10.436242143212567,11.079388738380151,11.725785333381955,12.37543192821799,13.028328522888248,13.684475117392733,14.343871711731435,15.006518305904368,15.672414899911523,16.341561493752906,17.013958087428513,17.689604680938345,18.36850127428239,19.050647867460672,19.736044460473178,20.424691053319904,21.116587646000855,21.81173423851604,22.510130830865442,23.211777423049067,23.91667401506691,24.624820606918984,25.336217198605286,26.050863790125813,26.76876038148056,27.489906972669537,28.214303563692738,28.941950154550163,29.672846745241813,30.406993335767673,31.144389926127776,31.885036516322096,32.628933106350644,33.37607969621341,34.12647628591041,34.88012287544163,35.63701946480707,36.397166054006746,37.16056264304064,37.92720923190876,38.697105820611085,39.470252409147655,40.24664899751845,41.02629558572347,41.80919217376271,42.59533876163618,43.38473534934386,44.17738193688578,44.973278524261914,45.77242511147228,46.57482169851687,47.38046828539569,48.18936487210871,49.00151145865599,49.81690804503747,50.63555463125319,51.45745121730312,52.28259780318727,53.11099438890567,53.94264097445827,54.7775375598451,55.61568414506616,56.457080730121454,57.30172731501096,58.14962389973469,59.000770484292644,59.855167068684835,60.71281365291124,61.57371023697186,62.43785682086673,63.3052534045958,64.17589998815912,65.04979657155661,65.9269431547884,66.80733973785436,67.69098632075456,68.577882903489,69.46802948605765,70.36142606846053,71.25807265069763,72.15796923276893,73.06111581467451,73.96751239641428,74.87715897798827],
"corriente":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,61.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,71.0,72.0,73.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0,81.0,82.0,83.0,84.0,85.0,86.0,87.0,88.0,89.0,90.0,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0]
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"yintercept":42.92929292929293,
"color":"#ee6666",
"linetype":"dashed",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"hline",
"data":{
}
},{
"mapping":{
"x":"corriente",
"y":"potencia"
},
"stat":"identity",
"color":"#fac858",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"corriente"
},{
"type":"float",
"column":"potencia"
}]
},
"spec_id":"8"
};
 var containerDiv = document.getElementById("VQJmOP");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1300.0,
 height: 500.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"

In [15]:
fun bisectionMethod(
    f: (Double) -> Double,
    a: Double,
    b: Double,
    tolerance: Double = 1e-6,
    maxIterations: Int = 100
): Double {
    require(f(a) * f(b) < 0) { "Function must have opposite signs at interval endpoints" }
    var left = a
    var right = b
    var iteration = 0
    var mid = 0.0
    while (iteration < maxIterations) {
        // Calculate midpoint
        mid = (left + right) / 2
        val fMid = f(mid)
        // Check if we're within tolerance
        if (right - left < 2 * tolerance || fMid == 0.0) return mid
        // Update interval
        if (f(left) * fMid < 0) right = mid else left = mid
        iteration++
    }
    return mid
}

val Imax_diodo = bisectionMethod(
    f = { v -> Pmax_diodo - calcular_potencia_conduccion_diodo(v.toFloat())},
    a = 0.0,
    b = 100.0
)
val Imax_igbt = bisectionMethod(
    f = { v -> Pmax_igbt - calcular_potencia_conduccion_igbt(v.toFloat()) - calcular_potencia_apagado_alt(v.toFloat()) - calcular_potencia_encendido_alt(v.toFloat()) },
    a = 0.0,
    b = 100.0
)
println("Potencia activa maxima impuesta por el Diodo: ${String.format("%.2f", 3*28*Imax_diodo/sqrt(2f))} W")
println("Potencia activa maxima impuesta por el IGBT: ${String.format("%.2f", 3*28*Imax_igbt/sqrt(2f))} W")

Potencia activa maxima impuesta por el Diodo: 5235,39 W
Potencia activa maxima impuesta por el IGBT: 3715,15 W
